# cuOpt link GPU test on Colab

Builds the GAMS/cuOpt link from a branch of this repository exactly like the x86_64 CI workflow, installs it into a fresh GAMS distribution and runs the GPU-dependent tests: a `trnsport` smoke test and `tests/test-gamslib-cuopt.py`.

1. **Runtime > Change runtime type > T4 GPU** (any NVIDIA GPU works).
2. Optional: store a GAMS license as Colab secret `GAMS_LICENSE` (key icon in the left sidebar, paste the content of `gamslice.txt`, enable notebook access). Without it GAMS runs in demo mode and larger gamslib models fail on size limits.
3. Set the branch below and run all cells.

In [ ]:
BRANCH = "claude/cuopt-wheels-pypi-j6l929"  # branch of GAMS-dev/cuoptlink-builder to build and test
CUDA = None    # 12 or 13; None picks the newest one the GPU driver supports
MODELS = []    # gamslib models to test, empty list runs the full baseline
RESLIM = 120   # time limit per model in seconds

In [ ]:
import os, re, shutil, subprocess, sys

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("No NVIDIA GPU found: select a GPU runtime (Runtime > Change runtime type)")
print(smi.stdout)
driver_cuda = int(re.search(r"CUDA Version:\s*(\d+)", smi.stdout).group(1))
if CUDA is None:
    CUDA = 13 if driver_cuda >= 13 else 12
if CUDA > driver_cuda:
    raise SystemExit(f"Driver supports CUDA {driver_cuda}, cannot run the CUDA {CUDA} build")
print(f"Testing the CUDA {CUDA} build")

In [ ]:
WORK = "/content/cuoptlink-builder"
shutil.rmtree(WORK, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                "https://github.com/GAMS-dev/cuoptlink-builder", WORK], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "patchelf", "pyyaml"], check=True)
print(subprocess.check_output(["git", "-C", WORK, "log", "-1", "--format=%h %s"], text=True))

## Build like CI

Runs the `run` steps of `.github/workflows/main-x86_64.yml` (venvs, GAMS download, compile, packaging) unchanged. `UV_PYTHON=3.13` makes the venvs match the `python3.13` paths the workflow uses. Takes a few minutes.

In [ ]:
import yaml

wf = yaml.safe_load(open(f"{WORK}/.github/workflows/main-x86_64.yml"))
env = dict(os.environ, UV_PYTHON="3.13")
for step in wf["jobs"]["build-link"]["steps"]:
    if "run" not in step or step.get("if") or step["name"].startswith("Install dependencies"):
        continue
    print("===", step["name"], flush=True)
    subprocess.run(["bash", "-e", "-c", step["run"]], cwd=WORK, env=env, check=True)

## Install into the GAMS distribution

In [ ]:
GAMSDIST = f"{WORK}/gamsdist"
for src in (f"{WORK}/release-cu{CUDA}", f"{WORK}/runtime-cu{CUDA}"):
    for name in os.listdir(src):
        shutil.copy2(os.path.join(src, name), GAMSDIST)
shutil.copy2(f"{GAMSDIST}/gamsconfig_cuopt.yaml", f"{GAMSDIST}/gamsconfig.yaml")

try:
    from google.colab import userdata
    lic = userdata.get("GAMS_LICENSE")
except Exception:
    lic = None
if lic:
    with open(f"{GAMSDIST}/gamslice.txt", "w") as f:
        f.write(lic.strip() + "\n")
    print("GAMS license installed")
else:
    print("No GAMS_LICENSE secret: GAMS runs in demo mode")
os.environ["PATH"] = GAMSDIST + os.pathsep + os.environ["PATH"]

## Smoke test: trnsport

In [ ]:
smoke = "/content/smoke"
shutil.rmtree(smoke, ignore_errors=True)
os.makedirs(smoke)
subprocess.run([f"{GAMSDIST}/gamslib", "-q", "trnsport"], cwd=smoke, check=True)
# GAMS ends normally even when the solver fails, so make a non-optimal solve abort the job
with open(f"{smoke}/trnsport.gms", "a") as f:
    f.write("\nabort$(transport.modelStat <> %modelStat.optimal%) 'cuOpt did not solve trnsport to optimality', transport.modelStat;\n")
r = subprocess.run([f"{GAMSDIST}/gams", "trnsport", "lp=cuopt", "lo=3"], cwd=smoke, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    raise SystemExit(f"trnsport failed with return code {r.returncode}")

## gamslib regression test

Solves the models in `tests/baseline.txt` with cuOpt and compares the objective with the CPLEX reference.

In [ ]:
r = subprocess.run([sys.executable, f"{WORK}/tests/test-gamslib-cuopt.py", "-g", GAMSDIST,
                    "-r", str(RESLIM), *MODELS], cwd="/content")
print("PASSED" if r.returncode == 0 else f"FAILED (exit code {r.returncode})")